In [6]:
import cv2
import os
import numpy as np
import torch
from pathlib import Path
from deep_sort_realtime.deepsort_tracker import DeepSort
from ultralytics import YOLO
import pandas as pd
from scipy.optimize import linear_sum_assignment
import glob

# Base path
basePath = "./Object_Tracking/"
task1ImagesPath = os.path.join(basePath, "Task1/images")
task1GtPath = os.path.join(basePath, "Task1/gt/gt.txt")
task2ImagesPath = os.path.join(basePath, "Task2/images")

# Define video paths for the helper function (needed for Cell 2)
task1_input_video_path = "task1_input.mp4"
task2_input_video_path = "task2_input.mp4"

In [7]:
# Cell 1.5: Utility Functions and Constants
FPS = 14  # Default frame rate


def images_to_video(images_dir, output_path, fps):
    # Helper to convert images to a video file
    image_files = sorted(glob.glob(os.path.join(images_dir, "*.jpg")))
    if not image_files:
        raise RuntimeError(f"No .jpg image files found in {images_dir}")

    first_img = cv2.imread(image_files[0])
    if first_img is None:
        raise RuntimeError(f"Could not read first image: {image_files[0]}")

    height, width = first_img.shape[:2]
    frame_size = (width, height)

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(output_path, fourcc, fps, frame_size)

    if not writer.isOpened():
        raise RuntimeError(f"VideoWriter could not be opened for {output_path}")

    for img_path in image_files:
        frame = cv2.imread(img_path)
        if frame is None:
            continue
        frame = cv2.resize(frame, frame_size)

        writer.write(frame)

    writer.release()


def computeIou(boxA, boxB):
    # boxA, boxB format: [l, t, w, h]
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[0] + boxA[2], boxB[0] + boxB[2])
    yB = min(boxA[1] + boxA[3], boxB[1] + boxB[3])
    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = boxA[2] * boxA[3]
    boxBArea = boxB[2] * boxB[3]
    return interArea / float(boxAArea + boxBArea - interArea)


def draw_and_log_box(frame, frame_idx, track):
    # For visualization/logging the track
    bbox = track.to_tlwh()  # (l, t, w, h)
    l, t, w, h = int(bbox[0]), int(bbox[1]), int(bbox[2]), int(bbox[3])
    r, b = l + w, t + h

    trackId = track.track_id

    cv2.rectangle(frame, (l, t), (r, b), (0, 255, 0), 2)
    cv2.putText(frame, str(trackId), (l, t - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

In [8]:
# Cell 2: Convert Task1 images to video
frameRate = FPS
task1_images_dir = task1ImagesPath

# Use the helper function to convert images to video
images_to_video(task1_images_dir, task1_input_video_path, frameRate)
images_to_video(task2ImagesPath, task2_input_video_path, frameRate)

# The rest is for printing (for reference)
images = sorted([img for img in os.listdir(task1ImagesPath) if img.endswith(".jpg")],
                key=lambda x: int(x.split('.')[0]))
print(images[:20])

['000001.jpg', '000002.jpg', '000003.jpg', '000004.jpg', '000005.jpg', '000006.jpg', '000007.jpg', '000008.jpg', '000009.jpg', '000010.jpg', '000011.jpg', '000012.jpg', '000013.jpg', '000014.jpg', '000015.jpg', '000016.jpg', '000017.jpg', '000018.jpg', '000019.jpg', '000020.jpg']


In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

yoloModel = YOLO("yolov8l.pt")
yoloModel.conf = 0.15
yoloModel.to(device)
tracker = DeepSort(max_age=45, n_init=3, max_iou_distance=0.7)

In [5]:
# Cell 4: Parameter Sweep and Evaluation with SAHI Tiled Inference 🏆

# 1. Define Sweep Ranges and Tiling Parameters
conf_sweep = [0.05, 0.06, 0.07, 0.08, 0.09, 0.10, 0.11, 0.12, 0.13, 0.14, 0.15, 0.16, 0.17, 0.18, 0.19, 0.20]
age_sweep = [20, 30, 45, 60]

slice_size = 1280
overlap_ratio = 0.2

# 2. Load Ground Truth (GT) Data
gtColumns = ["frame", "id", "bbLeft", "bbTop", "bbWidth", "bbHeight", "conf", "cls", "visibility"]
gtData = pd.read_csv(task1GtPath, sep=",", header=None, names=gtColumns)
gtData = gtData.iloc[:, :6]
allFrames = sorted(gtData['frame'].unique())

final_sweep_results = []
best_mota = -1.0
best_params = {}


def run_tracking_and_evaluate(confidence, max_age, n_init=3, max_iou_distance=0.7):
    tracker = DeepSort(max_age=max_age, n_init=n_init, max_iou_distance=max_iou_distance)
    trackingResults = []

    cap = cv2.VideoCapture(task1_input_video_path)
    if not cap.isOpened():
        return None

    # Update YOLO confidence threshold
    yoloModel.conf = confidence

    frameId = 1
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # --- YOLO Inference (no tiling) ---
        results = yoloModel(frame, conf=yoloModel.conf, verbose=False)[0]

        # Extract results for DeepSORT
        detections = []
        for box in results.boxes:
            if int(box.cls[0]) == 0:  # only person class
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                w = x2 - x1
                h = y2 - y1
                conf_score = float(box.conf[0])
                detections.append(([x1, y1, w, h], conf_score, "person"))

        # Update tracker
        tracks = tracker.update_tracks(detections, frame=frame)

        for track in tracks:
            if track.is_confirmed():
                l, t, w, h = track.to_tlwh()
                trackingResults.append([frameId, track.track_id, int(l), int(t), int(w), int(h)])

        frameId += 1

    cap.release()

    # --- Evaluation (MOTA + Avg IoU) remains unchanged ---
    fp, fn, idsw, gtCount = 0, 0, 0, 0
    total_matched_iou = 0.0
    total_matches = 0
    prevMatches = {}

    predData = np.array(trackingResults, dtype=float)

    for f in allFrames:
        gtFrame = gtData[gtData['frame'] == f][["id", "bbLeft", "bbTop", "bbWidth", "bbHeight"]].astype(float).values
        predFrame = predData[predData[:, 0] == f][:, 1:]

        gtCount += len(gtFrame)

        if len(gtFrame) == 0 or len(predFrame) == 0:
            fn += len(gtFrame)
            fp += len(predFrame)
            continue

        iouMatrix = np.zeros((len(gtFrame), len(predFrame)))
        for i, gtBox in enumerate(gtFrame):
            for j, predBox in enumerate(predFrame):
                iouMatrix[i, j] = computeIou(gtBox[1:], predBox[1:])

        gtIdx, predIdx = linear_sum_assignment(-iouMatrix)

        matchedGt, matchedPred = set(), set()
        for g, p in zip(gtIdx, predIdx):
            current_iou = iouMatrix[g, p]
            if current_iou >= 0.5:
                matchedGt.add(g)
                matchedPred.add(p)

                total_matched_iou += current_iou
                total_matches += 1

                if gtFrame[g, 0] in prevMatches and prevMatches[gtFrame[g, 0]] != predFrame[p, 0]:
                    idsw += 1
                prevMatches[gtFrame[g, 0]] = predFrame[p, 0]

        fn += len(gtFrame) - len(matchedGt)
        fp += len(predFrame) - len(matchedPred)

    mota = 1 - (fp + fn + idsw) / gtCount if gtCount > 0 else 0
    avg_iou = total_matched_iou / total_matches if total_matches > 0 else 0.0

    return mota, avg_iou, fp, fn, idsw, gtCount


# 3. Run the Sweep and Track Best Results
print("Starting Parameter Sweep for Task 1...")
for conf in conf_sweep:
    for age in age_sweep:
        mota, avg_iou, fp, fn, idsw, gtCount = run_tracking_and_evaluate(conf, age)

        current_result = {
            'conf': conf,
            'max_age': age,
            'MOTA': mota,
            'Avg_IoU': avg_iou,
            'FP': fp,
            'FN': fn,
            'IDSW': idsw
        }
        final_sweep_results.append(current_result)

        # Track the Best Parameters
        if mota > best_mota:
            best_mota = mota
            best_params = current_result.copy()

        print(f"Conf: {conf:.2f}, MaxAge: {age:d} -> MOTA: {mota:.4f}, Avg_IoU: {avg_iou:.4f}")

# 4. Display and Save Best Results
results_df = pd.DataFrame(final_sweep_results)
print("\n--- Sweep Results ---")
print(results_df.sort_values(by='MOTA', ascending=False))

# Save the best parameters to a file for 'future usage'
best_params_path = "best_task1_params.txt"
with open(best_params_path, 'w') as f:
    f.write(f"Best MOTA: {best_params['MOTA']:.4f}\n")
    f.write(f"Best Conf Threshold: {best_params['conf']:.2f}\n")
    f.write(f"Best DeepSORT Max Age: {best_params['max_age']}\n")
    f.write(f"SAHI Slice Size: {slice_size}\n")
    f.write(f"SAHI Overlap Ratio: {overlap_ratio}\n")

print(f"\n✅ Best parameters saved to {best_params_path} for use in Task 2.")

Starting Parameter Sweep for Task 1...
Conf: 0.05, MaxAge: 20 -> MOTA: 0.2771, Avg_IoU: 0.7422
Conf: 0.05, MaxAge: 30 -> MOTA: 0.2246, Avg_IoU: 0.7411
Conf: 0.05, MaxAge: 45 -> MOTA: 0.1715, Avg_IoU: 0.7405
Conf: 0.05, MaxAge: 60 -> MOTA: 0.1162, Avg_IoU: 0.7392
Conf: 0.06, MaxAge: 20 -> MOTA: 0.3057, Avg_IoU: 0.7436
Conf: 0.06, MaxAge: 30 -> MOTA: 0.2630, Avg_IoU: 0.7430
Conf: 0.06, MaxAge: 45 -> MOTA: 0.2122, Avg_IoU: 0.7418
Conf: 0.06, MaxAge: 60 -> MOTA: 0.1822, Avg_IoU: 0.7409
Conf: 0.07, MaxAge: 20 -> MOTA: 0.3020, Avg_IoU: 0.7475
Conf: 0.07, MaxAge: 30 -> MOTA: 0.2739, Avg_IoU: 0.7465
Conf: 0.07, MaxAge: 45 -> MOTA: 0.2311, Avg_IoU: 0.7445
Conf: 0.07, MaxAge: 60 -> MOTA: 0.1917, Avg_IoU: 0.7441
Conf: 0.08, MaxAge: 20 -> MOTA: 0.3128, Avg_IoU: 0.7505
Conf: 0.08, MaxAge: 30 -> MOTA: 0.2852, Avg_IoU: 0.7483
Conf: 0.08, MaxAge: 45 -> MOTA: 0.2461, Avg_IoU: 0.7474
Conf: 0.08, MaxAge: 60 -> MOTA: 0.2161, Avg_IoU: 0.7464
Conf: 0.09, MaxAge: 20 -> MOTA: 0.3172, Avg_IoU: 0.7512
Conf: 0.0

In [6]:
# Use best parameters from the sweep
best_conf = best_params.get('conf', 0.15)
best_max_age = best_params.get('max_age', 45)

# Initialize tracker and YOLO
tracker = DeepSort(max_age=best_max_age, n_init=3, max_iou_distance=0.7)
yoloModel.conf = best_conf

cap = cv2.VideoCapture(task1_input_video_path)
if not cap.isOpened():
    raise RuntimeError(f"Cannot open Task 1 input video: {task1_input_video_path}")

# Prepare video writer
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    fps = FPS

output_video_path = "task1.mp4"
task1_tracks_path = "task1_tracks.txt"
writer = cv2.VideoWriter(output_video_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

tracking_results = []  # To save frame,id,l,t,w,h

frame_idx = 1
while True:
    ret, frame = cap.read()
    if not ret:
        break

    # YOLOv8 inference
    results = yoloModel(frame, conf=yoloModel.conf, verbose=False)[0]

    detections = []
    for box in results.boxes:
        if int(box.cls[0]) == 0:  # only person class
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            w_box = x2 - x1
            h_box = y2 - y1
            conf_score = float(box.conf[0])
            detections.append(([x1, y1, w_box, h_box], conf_score, "person"))

    # Update DeepSORT
    tracks = tracker.update_tracks(detections, frame=frame)

    # Draw and log tracks
    for trk in tracks:
        if not trk.is_confirmed():
            continue
        l, t, w_box, h_box = trk.to_tlwh()
        tracking_results.append([frame_idx, trk.track_id, int(l), int(t), int(w_box), int(h_box)])
        # Draw bounding box
        cv2.rectangle(frame, (int(l), int(t)), (int(l + w_box), int(t + h_box)), (0, 255, 0), 2)
        cv2.putText(frame, str(trk.track_id), (int(l), int(t) - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    writer.write(frame)
    frame_idx += 1

cap.release()
writer.release()

# Save tracking results
pd.DataFrame(tracking_results, columns=["frame", "id", "bb_left", "bb_top", "bb_width", "bb_height"]) \
  .to_csv(task1_tracks_path, index=False)

print(f"\n✅ Task 1 annotated video saved at: {output_video_path}")
print(f"✅ Task 1 tracking results saved at: {task1_tracks_path}")



✅ Task 1 annotated video saved at: task1.mp4
✅ Task 1 tracking results saved at: task1_tracks.txt


In [ ]:
import cv2
import os
import numpy as np
import torch
from deep_sort_realtime.deepsort_tracker import DeepSort
import pandas as pd
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

best_conf = 0.25
best_max_age = 50
slice_size = 640
overlap_ratio = 0.35
MODEL_PATH = "yolov8l.pt"

task2_input_video_path = "task2_input.mp4"
task2_output_video_tiled = "task2_tiled_optimized.mp4"
counts_csv_path_tiled = "task2_counts_tiled.csv"
FPS = 14


def draw_and_log_box(frame, frame_idx, track):
    bbox = track.to_tlwh()  # (l, t, w, h)
    l, t, w, h = int(bbox[0]), int(bbox[1]), int(bbox[2]), int(bbox[3])
    r, b = l + w, t + h
    trackId = track.track_id
    cv2.rectangle(frame, (l, t), (r, b), (0, 255, 0), 2)
    cv2.putText(frame, str(trackId), (l, t - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)


if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_built():
    device = 'mps'
else:
    device = 'cpu'


print(f"{device}")
tracker = DeepSort(max_age=best_max_age, n_init=1, max_iou_distance=0.9, nms_max_overlap=0.5)

detection_model = AutoDetectionModel.from_pretrained(
    model_type='yolov8',
    model_path=MODEL_PATH,
    confidence_threshold=best_conf,
    device=device
)

cap2 = cv2.VideoCapture(task2_input_video_path)
if not cap2.isOpened():
    raise RuntimeError(f"Cannot open Task2 input video: {task2_input_video_path}")

w2 = int(cap2.get(cv2.CAP_PROP_FRAME_WIDTH))
h2 = int(cap2.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps2 = cap2.get(cv2.CAP_PROP_FPS)
if fps2 <= 0: fps2 = FPS

writer2 = cv2.VideoWriter(task2_output_video_tiled, cv2.VideoWriter_fourcc(*'mp4v'), fps2, (w2, h2))

frame_idx = 1
frame_counts = []

print(f"Starting Tiled (SAHI) Inference on Task 2. Conf={best_conf}, MaxAge={best_max_age}")

while True:
    ok, frame = cap2.read()
    if not ok:
        break

    sliced_results = get_sliced_prediction(
        frame,
        detection_model,
        slice_height=slice_size,
        slice_width=slice_size,
        overlap_height_ratio=overlap_ratio,
        overlap_width_ratio=overlap_ratio,
        perform_standard_pred=True,
        verbose=0
    )

    detections = []
    for detection in sliced_results.object_prediction_list:
        if detection.category.id == 0:
            x1, y1, x2, y2 = detection.bbox.to_xyxy()
            w = x2 - x1
            h = y2 - y1
            conf_score = detection.score.value
            if h > (h2 * 0.6):
                continue

            detections.append(([x1, y1, w, h], conf_score, "person"))

    tracks = tracker.update_tracks(detections, frame=frame)

    people_in_frame = sum([1 for trk in tracks if trk.is_confirmed()])
    frame_counts.append((frame_idx, people_in_frame))

    for trk in tracks:
        if not trk.is_confirmed():
            continue
        draw_and_log_box(frame, frame_idx, trk)

    writer2.write(frame)

    if frame_idx % 50 == 0:
        print(f"Frame {frame_idx} processed. Count: {people_in_frame}")

    frame_idx += 1

cap2.release()
writer2.release()

pd.DataFrame(frame_counts, columns=["Number", "Count"]).to_csv(counts_csv_path_tiled, index=False)

print("\nFinished Task 2 tracking with Tiled (SAHI) Inference.")
print(f"Annotated Task 2 video saved at: {task2_output_video_tiled}")
print(f"Count data saved at: {counts_csv_path_tiled}")

mps
Starting Tiled (SAHI) Inference on Task 2. Conf=0.25, MaxAge=50
